In [13]:
from DataSynthesizer.DataDescriber import DataDescriber
from DataSynthesizer.DataGenerator import DataGenerator
from DataSynthesizer.ModelInspector import ModelInspector
from DataSynthesizer.lib.utils import read_json_file, display_bayesian_network

import sys
import pandas as pd
import datetime
import pickle

In [14]:
import os
import sys
import numpy as np
import subprocess
from shutil import which

In [16]:
# 0..21 から 21(=index 20) を除くチーム一覧
# prep: 1-22 (21 is missing)
# contest: 1-20,22-24
def get_teams():
    arr = np.arange(1, 25)
    return np.delete(arr, 20)  # 21 をスキップ（コンテスト仕様に合わせる）

In [ ]:
def loop_for_all_teams(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...] のようなリスト
    dry_run: True -> 実行せず展開コマンドのみ表示
    strict: True -> {id:02d} が一つも無ければ例外
    continue_on_error: True -> 失敗しても次の team へ。False -> そこで中断
    cwd: サブプロセスの作業ディレクトリ（attack/ ディレクトリの相対パス解決に使える）
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    # 'python' を実行ファイルに置換（環境ズレ回避）
    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    # 事前: 実行ファイルの存在チェック（python 以外の最初の実体コマンドにも対応）
    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        # 簡易プリフライト: 既知の入力系ファイルっぽい引数を存在確認
        # （.csv, .json かつ -o/--out* ではない位置を対象にする）
        def is_out_flag(i):
            return isinstance(cmd[i-1], str) and (
                cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or cmd[i-1].startswith("--out")
            )

        missing_inputs = []
        for i, a in enumerate(cmd):
            if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                if not is_out_flag(i):  # 出力ではなく入力と推定
                    apath = a if cwd is None else os.path.join(cwd, a)
                    if not os.path.exists(apath):
                        missing_inputs.append(a)

        print(">>", " ".join(cmd))
        if missing_inputs:
            msg = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", msg)
                continue
            else:
                raise FileNotFoundError(msg)

        if dry_run:
            continue

        try:
            # 標準出力・標準エラーを取得して、失敗時に見せる
            completed = subprocess.run(
                cmd, check=True, cwd=cwd,
                capture_output=True, text=True
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
            if e.stdout:
                print("--- stdout ---")
                print(e.stdout.strip())
            if e.stderr:
                print("--- stderr ---")
                print(e.stderr.strip())
            if not continue_on_error:
                raise

In [ ]:
# team=22 のみで実行するバージョン(テスト用)
def loop_for_team_22(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    指定コマンドを team=22 のみで実行・検証する
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    team = 22
    cmd = cmd0[:]
    for ind in id_indices:
        cmd[ind] = cmd[ind].format(id=team)

    def is_out_flag(i):
        return isinstance(cmd[i-1], str) and (
            cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
            or cmd[i-1].startswith("--out")
        )

    missing_inputs = []
    for i, a in enumerate(cmd):
        if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
            if not is_out_flag(i):
                apath = a if cwd is None else os.path.join(cwd, a)
                if not os.path.exists(apath):
                    missing_inputs.append(a)

    print(">>", " ".join(cmd))
    if missing_inputs:
        msg = f"[team {team}] Missing input files: {missing_inputs}"
        if continue_on_error:
            print("!!", msg)
        else:
            raise FileNotFoundError(msg)

    if dry_run:
        return

    try:
        completed = subprocess.run(
            cmd, check=True, cwd=cwd,
            capture_output=True, text=True
        )
        if completed.stdout:
            print(completed.stdout.strip())
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
        if e.stdout:
            print("--- stdout ---")
            print(e.stdout.strip())
        if e.stderr:
            print("--- stderr ---")
            print(e.stderr.strip())
        if not continue_on_error:
            raise

In [18]:
# 必要なら cwd='プロジェクトのルート' を指定（例: cwd=r'c:\work\pwscup2025'）
cwd = r"/home/kikuchih/pwscup2025-scripts"  # <- 適宜書き換え
os.chdir(cwd)

In [19]:
## Create input folder and Place "Original Datasets(B**/BB**)".
### create "in/"
### place B22_1, etc. in "in/"
if not os.path.exists("in"):
    os.makedirs("in")

In [20]:
prep_original = "B"
contest_original = "BB"
prep_anon = "C"
contest_anon = "CC"
prep_model = "D"
contest_model = "DD"

In [21]:
# choose mode(prep or contest)
mode = "contest"

if mode == "prep":
    mode_original = prep_original
    mode_anon = prep_anon
    mode_model = prep_model
else:
    mode_original = contest_original
    mode_anon = contest_anon
    mode_model = contest_model

In [23]:
## Creation of Ci with original samples
Ci_anonymization_original = ["python", "anonymization/ano.py", f"in/{mode_original}"+"{id:02d}.csv", f"in/{mode_anon}"+"{id:02d}.csv", "--seed", "42"]
loop_for_all_teams(Ci_anonymization_original)
print("sample Ci-anonymization completed")

>> /home/kikuchih/miniconda3/envs/pwscup2024/bin/python anonymization/ano.py in/BB01.csv in/CC01.csv --seed 42


FileNotFoundError: [team 1] Missing input files: ['in/BB01.csv', 'in/CC01.csv']

In [ ]:
## Creation of Di with original samples(Bi only)
Di_anonymization_original_Bi = ["python", "analysis/xgbt_train.py", "in/{mode_original}{id:02d}.csv", "--model-json", "in/{mode_model}{id:02d}_Bi.json"]
loop_for_all_teams(Di_anonymization_original_Bi)
print("sample Di-anonymization(Bi only) completed")


In [ ]:
## Creation of Di with original samples(Ci only)
Di_anonymization_original_Ci = ["python", "analysis/xgbt_train.py", "in/{mode_anon}{id:02d}.csv", "--model-json", "in/{mode_model}{id:02d}_Ci.json"]
loop_for_all_teams(Di_anonymization_original_Ci)
print("sample Di-anonymization(Ci only) completed")

In [ ]:
## Creation of Di with original samples(Bi and Ci)
Di_anonymization_original = ["python", "anonymization/gen_Di.py", "in/{mode_original}{id:02d}.csv", "in/{mode_anon}{id:02d}.csv"]
loop_for_all_teams(Di_anonymization_original)
print("sample Ci-anonymization completed")